# CXR-Classifier: Interactive Exploration

This notebook walks through the full pipeline: configuration, data loading, model creation, training (a single epoch on a tiny subset for demo), evaluation, and Grad-CAM visualization.

Run on the real dataset by setting `CONFIG_PATH = "configs/config.yaml"`. The default here uses the synthetic fixture shipped in `tests/conftest.py` so the notebook runs anywhere without the 5 GB Kaggle download.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))
sys.path.insert(0, str(Path.cwd() / "tests"))  # for conftest fixtures

import numpy as np
import torch

from cxr_classifier.config import Config, load_config
from cxr_classifier.data import get_dataloaders
from cxr_classifier.models import create_model, get_model_info
from cxr_classifier.evaluation import Evaluator, GradCAM, generate_gradcam_grid

## 1. Load config

By default we read `configs/config.yaml`. Override with `CONFIG_PATH` to point at the full-dataset config.

In [ ]:
CONFIG_PATH = "configs/config.yaml"  # or your custom config
config = load_config(CONFIG_PATH)
print(f"Model:      {config.model.name}")
print(f"Classes:    {config.dataset.classes}")
print(f"Image size: {config.dataset.image_size}")
print(f"Batch size: {config.dataset.batch_size}")
print(f"Epochs:     {config.training.epochs}")
print(f"Optimizer:  {config.training.optimizer.name} @ lr={config.training.optimizer.lr}")

## 2. Build dataloaders

This requires the dataset to exist at `config.dataset.data_root/{train,val,test}/<class>/*.jpg`. For a real run, point `data_root` at the Kaggle dataset folder.

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(config)
print(f"Train batches: {len(train_loader)}, samples: {len(train_loader.dataset)}")
print(f"Val batches:   {len(val_loader)}, samples: {len(val_loader.dataset)}")
print(f"Test batches:  {len(test_loader)}, samples: {len(test_loader.dataset)}")

# Show one batch
xb, yb = next(iter(train_loader))
print(f"\nBatch x: {xb.shape} {xb.dtype}, y: {yb.shape} {yb.dtype}")
print(f"y distribution: {np.bincount(yb.numpy(), minlength=config.dataset.num_classes)}")

## 3. Create the model

`create_model` wraps `timm.create_model` and replaces the classifier head with a 512-unit bottleneck + dropout.

In [ ]:
model = create_model(config)
info = get_model_info(model)
print(f"Total params:     {info['total_parameters']:,}")
print(f"Trainable params: {info['trainable_parameters']:,}")
print(f"Size:             {info['model_size_mb']:.1f} MB")

# Smoke forward
model.eval()
with torch.no_grad():
    out = model(xb[:2])
print(f"\nOutput shape: {out.shape} (expect [batch, num_classes])")

## 4. Evaluation on a checkpoint

If you've already trained and have a checkpoint, you can load it and evaluate. Skip this cell on a fresh checkout.

In [ ]:
from pathlib import Path

CKPT = Path("outputs/best_model.pth")
if CKPT.exists():
    ckpt = torch.load(CKPT, map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print(f"Loaded {CKPT} (epoch {ckpt.get('epoch', '?')})")

    from cxr_classifier.evaluation import evaluate_model
    results = evaluate_model(
        model=model,
        dataloader=test_loader,
        device=torch.device("cpu"),
        class_names=config.evaluation.class_names,
        save_dir="outputs/evaluation",
    )
else:
    print(f"No checkpoint at {CKPT} — skipping evaluation. Train first to populate outputs/.")

## 5. Grad-CAM visualization

Runs on the test set and saves a grid of heatmap overlays — one correctly-classified sample per class.

In [ ]:
if CKPT.exists():
    out = generate_gradcam_grid(
        model=model,
        dataset=test_loader.dataset,
        device=torch.device("cpu"),
        class_names=config.evaluation.class_names,
        save_dir="outputs/gradcam",
        num_samples=4,
    )
    print(f"Grad-CAM grid saved to {out}")
else:
    print("Skip Grad-CAM until a checkpoint exists.")

## 6. Next steps

- `python scripts/train.py --config configs/config.yaml` to train end-to-end
- `tensorboard --logdir logs` to monitor training
- `python scripts/evaluate.py --checkpoint outputs/best_model.pth` to re-run evaluation
- Swap models in `configs/config.yaml` (e.g. `convnext_tiny`, `efficientnet_b3`, `swin_tiny_patch4_window7_224`)